# 07 - AgentCore Memory
Create an AgentCore Memory store, configure extraction strategies (semantic, user-preference, summary), and test session read/write with long-term search.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
import time
from pathlib import Path
from cleanup_utils import ResourceTracker

config = json.loads(Path("config.json").read_text())
REGION = config["REGION"]
PREFIX = config["PROJECT_PREFIX"]

tracker = ResourceTracker()
print(f"\u2713 Config loaded (region={REGION}, prefix={PREFIX})")

## Step 2: Initialize Memory Client

Connect to the AgentCore Memory service.

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.session import MemorySessionManager
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole

memory_client = MemoryClient(region_name=REGION)
print("\u2713 Memory client initialized")

## Step 3: Create Memory Store

Create a memory store with multiple extraction strategies.

In [ ]:
MEMORY_NAME = f"{PREFIX}-agent-memory"

# Check if memory store already exists
existing_stores = memory_client.list_memory_stores()
existing = next(
    (s for s in existing_stores if s["name"] == MEMORY_NAME),
    None
)

if existing:
    memory_id = existing["id"]
    print(f"\u2713 Memory store already exists: {memory_id}")
else:
    print(f"\u231b Creating memory store: {MEMORY_NAME}...")
    response = memory_client.create_memory_store(
        name=MEMORY_NAME,
        description="OrgIQ agent memory for conversation context and user preferences",
        strategies=[
            {
                "type": "SEMANTIC",
                "name": "semantic-facts",
                "description": "Extract factual information about organizational knowledge"
            },
            {
                "type": "USER_PREFERENCE",
                "name": "user-preferences",
                "description": "Track user preferences for search results and communication style"
            },
            {
                "type": "SUMMARY",
                "name": "conversation-summaries",
                "description": "Summarize conversations for long-term context"
            }
        ]
    )
    memory_id = response["id"]
    tracker.add_agentcore_memory(memory_id, MEMORY_NAME)
    print(f"\u2713 Memory store created: {memory_id}")

## Step 4: Create a Session

Sessions group related interactions together.

In [ ]:
import uuid

session_id = str(uuid.uuid4())
user_id = "user-alice-chen"

session_manager = MemorySessionManager(
    memory_client=memory_client,
    memory_store_id=memory_id
)

session = session_manager.create_session(
    session_id=session_id,
    user_id=user_id,
    metadata={"source": "notebook", "agent": "orgiq-search"}
)

print(f"\u2713 Session created")
print(f"  Session ID: {session_id}")
print(f"  User ID: {user_id}")

## Step 5: Write Conversation Messages to Memory

Simulate a conversation and persist it to the memory store.

In [ ]:
# Simulate a multi-turn conversation
conversation = [
    ConversationalMessage(
        role=MessageRole.USER,
        content="Who are the best machine learning experts in our org?"
    ),
    ConversationalMessage(
        role=MessageRole.ASSISTANT,
        content="Based on the knowledge graph, Bob Kumar is our top ML expert with a strength score of 0.95. He's a Staff ML Engineer and has authored the ML Pipeline Architecture document."
    ),
    ConversationalMessage(
        role=MessageRole.USER,
        content="I prefer seeing results sorted by recency rather than score. Also, I work on the platform team."
    ),
    ConversationalMessage(
        role=MessageRole.ASSISTANT,
        content="Got it! I'll prioritize recent contributions when showing results. I've noted that you're on the platform team - I'll factor that into recommendations for collaboration."
    ),
    ConversationalMessage(
        role=MessageRole.USER,
        content="Can you find documentation about setting up CI/CD pipelines?"
    ),
    ConversationalMessage(
        role=MessageRole.ASSISTANT,
        content="I found Dave Park's 'GitOps Pipeline Setup' document which covers CI/CD extensively. Dave is on your team (Platform) and has a 0.9 strength in CI/CD."
    ),
]

# Write messages to memory
session_manager.add_messages(
    session_id=session_id,
    messages=conversation
)

print(f"\u2713 {len(conversation)} messages written to memory")

## Step 6: Trigger Memory Extraction

Process the conversation to extract facts, preferences, and summaries.

In [ ]:
print("\u231b Triggering memory extraction...")
extraction = memory_client.extract_memories(
    memory_store_id=memory_id,
    session_id=session_id
)

# Wait for extraction to complete
time.sleep(5)

print(f"\u2713 Memory extraction triggered")
print(f"  Extraction ID: {extraction.get('extractionId', 'N/A')}")

## Step 7: Read Session History

Retrieve the conversation history from the session.

In [ ]:
# Read back the session messages
history = session_manager.get_messages(
    session_id=session_id,
    limit=10
)

print(f"Session history ({len(history)} messages):")
for msg in history:
    role = msg.role.value if hasattr(msg.role, 'value') else msg.role
    content_preview = msg.content[:80] if len(msg.content) > 80 else msg.content
    print(f"  [{role}] {content_preview}...")

print("\n\u2713 Session history retrieved")

## Step 8: Search Long-Term Memory

Search across extracted memories using semantic search.

In [ ]:
# Semantic search across long-term memory
search_results = memory_client.search_memories(
    memory_store_id=memory_id,
    query="machine learning expertise",
    user_id=user_id,
    limit=5
)

print("Memory search: 'machine learning expertise'")
print(f"Found {len(search_results)} memories:")
for mem in search_results:
    print(f"  - [{mem.get('strategy', 'unknown')}] {mem.get('content', '')[:100]}")
    print(f"    Score: {mem.get('score', 'N/A')}")

print("\n\u2713 Long-term memory search working")

## Step 9: Search User Preferences

In [ ]:
# Search specifically for user preferences
pref_results = memory_client.search_memories(
    memory_store_id=memory_id,
    query="sorting and display preferences",
    user_id=user_id,
    strategy_name="user-preferences",
    limit=5
)

print("User preference memories:")
for mem in pref_results:
    print(f"  - {mem.get('content', '')[:150]}")

print("\n\u2713 User preference extraction working")

## Step 10: Create a Second Session (Same User)

Demonstrate memory persistence across sessions.

In [ ]:
# New session for the same user
session_id_2 = str(uuid.uuid4())
session_2 = session_manager.create_session(
    session_id=session_id_2,
    user_id=user_id,
    metadata={"source": "notebook", "agent": "orgiq-search", "session_number": 2}
)

# Search memories from previous session
context_memories = memory_client.search_memories(
    memory_store_id=memory_id,
    query="What does this user prefer?",
    user_id=user_id,
    limit=3
)

print(f"New session: {session_id_2}")
print(f"\nMemories from previous sessions:")
for mem in context_memories:
    print(f"  - {mem.get('content', '')[:120]}")

print("\n\u2713 Cross-session memory persistence verified")

## Step 11: List All Sessions for User

In [ ]:
# List all sessions for this user
sessions = session_manager.list_sessions(user_id=user_id)

print(f"Sessions for user '{user_id}':")
for sess in sessions:
    print(f"  - {sess['session_id'][:12]}... (created: {sess.get('created_at', 'N/A')})")

print(f"\n\u2713 {len(sessions)} sessions found")

## Step 12: Memory Store Statistics

In [ ]:
# Get memory store info
store_info = memory_client.get_memory_store(memory_store_id=memory_id)

print(f"Memory Store: {store_info['name']}")
print(f"  ID: {store_info['id']}")
print(f"  Status: {store_info.get('status', 'N/A')}")
print(f"  Strategies:")
for strategy in store_info.get("strategies", []):
    print(f"    - {strategy['name']} ({strategy['type']})")

print("\n\u2713 Memory store info retrieved")

## Step 13: Save Configuration

In [ ]:
config["MEMORY_ID"] = memory_id
Path("config.json").write_text(json.dumps(config, indent=2))

print(f"\u2713 Config saved")
print(f"  MEMORY_ID: {memory_id}")

## Summary

- [x] AgentCore Memory store created with 3 strategies
- [x] Semantic extraction: factual knowledge captured
- [x] User preference extraction: display/sorting preferences tracked
- [x] Summary extraction: conversation summaries generated
- [x] Session creation and message persistence working
- [x] Long-term memory semantic search working
- [x] Cross-session memory persistence verified
- [x] Multi-session tracking per user

**Next:** [08-agentcore-gateway.ipynb](08-agentcore-gateway.ipynb) — Create an MCP Gateway with tool targets